# Phase 1 — ML emulator of FLEXPART's convection scheme

This notebook builds, trains and validates a two-model emulator of the convection
redistribution matrix used in FLEXPART (Emanuel & Živković-Rothman scheme,
Forster et al. 2007).

**Pipeline**

1. Atmospheric columns (T, q profiles on 27 pressure levels) — synthetic by default, or ERA5.
2. Teacher scheme → redistribution matrix `M[i, j]` = fraction of air at level *i* moved to level *j*.
3. Model 1 (**gate**): is convection active in this column?
4. Model 2 (**redistribution net**): predicts `M`; a row-wise softmax guarantees every row is
   non-negative and sums to 1 (mass conservation built in).
5. Validation: gate F1, matrix error, mass-flux profile, particle test, speed.
6. Export to TorchScript for Phase 2 (coupling into FLEXPART's Fortran code via FTorch).

> ⚠️ **The teacher scheme in Section 3 is a simplified STAND-IN**, so the whole pipeline runs
> today. It mimics the structure of convective redistribution (sub-cloud air lifted to
> detrainment levels, compensating subsidence) but it is **not** FLEXPART's scheme.
> Section 9 explains how to replace it with the real Fortran routine. Only that one function changes.

**How to run:** `Runtime → Run all`. On Colab, a GPU runtime is optional
(`Runtime → Change runtime type → T4 GPU`); everything also runs on CPU.

## 0. Setup

In [ ]:
import os, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="torch.jit")  # TorchScript is still what FTorch loads

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---- Configuration -------------------------------------------------------
CFG = {
    "data_source": "synthetic",   # "synthetic" (runs anywhere) or "era5" (needs a CDS API key)
    "n_columns": int(os.environ.get("N_COLUMNS", 60000)),  # synthetic columns to generate
    "mount_drive": True,          # save data and checkpoints to Google Drive (Colab only)
    "seed": 42,
    "gate_epochs": int(os.environ.get("GATE_EPOCHS", 15)),
    "redist_epochs": int(os.environ.get("REDIST_EPOCHS", 40)),
    "batch_size": 256,
    "lr": 1e-3,
}

if IN_COLAB and CFG["mount_drive"]:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/flexpart_phase1"
else:
    OUT_DIR = os.path.abspath("flexpart_phase1_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Colab: {IN_COLAB} | device: {DEVICE} | outputs: {OUT_DIR}")

# ERA5 pressure levels (hPa), surface → top
P_LEVELS = np.array([1000, 975, 950, 925, 900, 875, 850, 825, 800, 775, 750,
                     700, 650, 600, 550, 500, 450, 400, 350, 300, 250,
                     225, 200, 175, 150, 125, 100], dtype=np.float64)
NLEV = len(P_LEVELS)

# Layer air mass per level ∝ pressure thickness (used for weights and flux diagnostics)
_edges = np.concatenate([[P_LEVELS[0] + (P_LEVELS[0] - P_LEVELS[1]) / 2],
                         (P_LEVELS[:-1] + P_LEVELS[1:]) / 2,
                         [P_LEVELS[-1] - (P_LEVELS[-2] - P_LEVELS[-1]) / 2]])
LAYER_DP = _edges[:-1] - _edges[1:]          # hPa, all positive
LAYER_W = LAYER_DP / LAYER_DP.sum()          # mass weight of each level
print(f"{NLEV} levels, {P_LEVELS[0]:.0f}–{P_LEVELS[-1]:.0f} hPa")

## 1. Thermodynamics helpers
Parcel ascent (dry adiabat → moist pseudo-adiabat), CAPE, CIN, LCL and level of neutral
buoyancy (LNB). Vectorised over many columns at once.

In [ ]:
RD, RV, CP, LV, G = 287.04, 461.5, 1004.6, 2.501e6, 9.81
EPS = RD / RV

def sat_vapour_pressure(T):
    """Bolton (1980), T in K → Pa."""
    return 611.2 * np.exp(17.67 * (T - 273.15) / (T - 29.65))

def q_sat(T, p_pa):
    es = sat_vapour_pressure(T)
    return EPS * es / (p_pa - (1 - EPS) * es)

def r_sat(T, p_pa):
    es = sat_vapour_pressure(T)
    return EPS * es / np.maximum(p_pa - es, 1.0)

def virtual_temp(T, q):
    return T * (1 + 0.608 * q)

def parcel_diagnostics(T, q, p_hpa=P_LEVELS, nsub=10):
    """Lift a surface parcel through each column.

    T, q: (N, NLEV) arrays. Returns dict with parcel T, buoyancy, CAPE, CIN,
    LCL index/pressure and LNB index (-1 if no positive buoyancy).
    """
    N, n = T.shape
    p = p_hpa * 100.0
    Tp = np.empty_like(T)
    Tp[:, 0] = T[:, 0]
    q0 = q[:, 0].copy()
    Tcur = T[:, 0].copy()
    sat = q0 >= q_sat(Tcur, p[0])
    lcl_idx = np.where(sat, 0, n - 1)
    for k in range(1, n):
        p_prev = p[k - 1]
        for s in range(1, nsub + 1):
            p_new = p[k - 1] * (p[k] / p[k - 1]) ** (s / nsub)
            dp = p_new - p_prev                     # negative (pressure falls)
            # dry adiabat for unsaturated parcels
            T_dry = Tcur * (p_new / p_prev) ** (RD / CP)
            # moist pseudo-adiabat for saturated parcels
            rs = r_sat(Tcur, p_prev)
            dTdp = (RD * Tcur + LV * rs) / (p_prev * (CP + LV**2 * rs * EPS / (RD * Tcur**2)))
            T_moist = Tcur + dTdp * dp
            Tcur = np.where(sat, T_moist, T_dry)
            newly = (~sat) & (q0 >= q_sat(Tcur, p_new))
            lcl_idx = np.where(newly & (lcl_idx == n - 1), k, lcl_idx)
            sat = sat | newly
            p_prev = p_new
        Tp[:, k] = Tcur
    qp = np.where(np.arange(n)[None, :] >= lcl_idx[:, None],
                  q_sat(Tp, p[None, :]), q0[:, None])
    B = virtual_temp(Tp, qp) - virtual_temp(T, q)           # K
    dlnp = np.log(p[:-1] / p[1:])                            # positive
    layer = RD * 0.5 * (B[:, :-1] + B[:, 1:]) * dlnp[None, :]  # J/kg per layer
    cape = np.clip(layer, 0, None).sum(axis=1)
    pos = layer > 0
    first_pos = np.where(pos.any(axis=1), pos.argmax(axis=1), n - 1)
    below_lfc = np.arange(n - 1)[None, :] < first_pos[:, None]
    cin = np.where(below_lfc, np.clip(layer, None, 0), 0).sum(axis=1)
    has_pos = (B > 0)
    lnb_idx = np.where(has_pos.any(axis=1), n - 1 - has_pos[:, ::-1].argmax(axis=1), -1)
    return {"Tp": Tp, "B": B, "cape": cape, "cin": cin, "lcl_idx": lcl_idx,
            "lcl_p": p_hpa[lcl_idx], "lnb_idx": lnb_idx}

## 2. Atmospheric columns

**Synthetic (default):** realistic soundings from four regimes (tropical ocean, monsoon land,
mid-latitude, dry/desert), so the pipeline runs without any download.

**ERA5:** set `CFG["data_source"] = "era5"`, put your Climate Data Store key in
`~/.cdsapirc`, and run the download cell. The ERA5 loader is written for the current CDS
netCDF layout but was not tested in this notebook's preparation (no CDS access); check the
variable and dimension names printed by `ds` on first use.

In [ ]:
REGIMES = {  # name: (probability, surface T range K, surface RH range, lapse rate K/km)
    "tropical_ocean": (0.35, (298, 304), (0.75, 0.95), (5.0, 6.5)),
    "monsoon_land":   (0.25, (300, 310), (0.50, 0.90), (6.0, 8.0)),
    "midlatitude":    (0.25, (275, 295), (0.50, 0.90), (5.5, 7.5)),
    "dry":            (0.15, (290, 312), (0.10, 0.40), (7.0, 9.5)),
}

def synthetic_columns(N, rng):
    names = list(REGIMES)
    probs = np.array([REGIMES[k][0] for k in names])
    reg = rng.choice(len(names), size=N, p=probs / probs.sum())
    U = lambda lohi, m: rng.uniform(lohi[0], lohi[1], size=m)
    Ts, RHs, gam = np.empty(N), np.empty(N), np.empty(N)
    for i, k in enumerate(names):
        m = reg == i
        _, tr, rr, gr = REGIMES[k]
        Ts[m], RHs[m], gam[m] = U(tr, m.sum()), U(rr, m.sum()), U(gr, m.sum())
    p = P_LEVELS[None, :]
    T = Ts[:, None] * (p / 1000.0) ** (RD * (gam[:, None] / 1000.0) / G)
    T_trop = rng.uniform(190, 215, size=N)[:, None]
    T = np.maximum(T, T_trop) + rng.normal(0, 0.5, size=T.shape)
    H_rh = rng.uniform(250, 600, size=N)[:, None]
    A = rng.uniform(-0.3, 0.3, size=N)[:, None]
    pc = rng.uniform(400, 750, size=N)[:, None]
    RH = RHs[:, None] * np.exp(-(1000.0 - p) / H_rh) + A * np.exp(-((p - pc) / 80.0) ** 2)
    RH = np.clip(RH, 0.02, 1.0)
    q = RH * q_sat(T, p * 100.0)
    ps = rng.uniform(1000, 1020, size=N)
    return T, q, ps, reg

def download_era5(path, year="2019", months=("01", "04", "07", "10"),
                  area=(10, 60, -10, 100)):   # N, W, S, E: tropical Indian Ocean
    """Download ERA5 T and q on pressure levels (needs cdsapi + ~/.cdsapirc)."""
    import cdsapi
    c = cdsapi.Client()
    c.retrieve("reanalysis-era5-pressure-levels", {
        "product_type": ["reanalysis"],
        "variable": ["temperature", "specific_humidity"],
        "year": [year], "month": list(months),
        "day": ["01", "08", "15", "22"], "time": ["00:00", "06:00", "12:00", "18:00"],
        "pressure_level": [str(int(x)) for x in P_LEVELS],
        "data_format": "netcdf", "area": list(area),
    }, path)

def load_era5_columns(path):
    import xarray as xr
    ds = xr.open_dataset(path)
    print(ds)
    lev = "pressure_level" if "pressure_level" in ds.dims else "level"
    tdim = "valid_time" if "valid_time" in ds.dims else "time"
    ds = ds.sel({lev: P_LEVELS})
    T = ds["t"].transpose(tdim, "latitude", "longitude", lev).values.reshape(-1, NLEV)
    q = ds["q"].transpose(tdim, "latitude", "longitude", lev).values.reshape(-1, NLEV)
    times = np.repeat(ds[tdim].values, ds.sizes["latitude"] * ds.sizes["longitude"])
    ps = np.full(len(T), 1013.0)  # add ERA5 single-level surface pressure for real use
    return T.astype(np.float64), q.astype(np.float64), ps, times

rng = np.random.default_rng(CFG["seed"])
if CFG["data_source"] == "era5":
    era5_file = os.path.join(OUT_DIR, "era5_columns.nc")
    if not os.path.exists(era5_file):
        download_era5(era5_file)
    T_all, q_all, ps_all, times_all = load_era5_columns(era5_file)
    regime_all = np.zeros(len(T_all), dtype=int)
else:
    T_all, q_all, ps_all, regime_all = synthetic_columns(CFG["n_columns"], rng)
    times_all = None
print(f"Columns: {T_all.shape[0]:,} × {NLEV} levels")

## 3. Teacher scheme → redistribution matrices

⚠️ **STAND-IN for FLEXPART's convection routine** (replace in Section 9).

For a convective column (CAPE > 300 J/kg, CIN > −50 J/kg, cloud at least 3 levels deep):
- each sub-cloud level sends a fraction *f* (grows with CAPE) of its air up to detrainment
  levels (70 % peaked just below the LNB, 30 % spread through the cloud layer);
- cloud-layer levels subside one level, sized to roughly balance the lifted mass.

Every row of the matrix sums to 1. Non-convective columns get the identity matrix.

In [ ]:
def teacher_standin(T, q, ps=None, p_hpa=P_LEVELS):
    diag = parcel_diagnostics(T, q, p_hpa)
    N, n = T.shape
    lcl, lnb, cape, cin = diag["lcl_idx"], diag["lnb_idx"], diag["cape"], diag["cin"]
    conv = (cape > 300) & (cin > -50) & (lnb >= lcl + 3)
    M = np.broadcast_to(np.eye(n), (N, n, n)).copy()
    lev = np.arange(n)
    for c in np.where(conv)[0]:
        L, T_ = lcl[c], lnb[c]
        f = np.clip(cape[c] / 4000.0, 0.05, 0.6)
        cloud = (lev > L) & (lev <= T_)
        gauss = np.exp(-0.5 * ((lev - (T_ - 1)) / 2.0) ** 2) * cloud
        d = 0.7 * gauss / gauss.sum() + 0.3 * cloud / cloud.sum()
        src = lev <= L
        U = f * LAYER_DP[src].sum()
        s = min(0.5, U / LAYER_DP[cloud].sum())
        for i in lev[src]:
            M[c, i, :] = (1 - f) * (lev == i) + f * d
        for i in lev[cloud]:
            M[c, i, :] = 0.0
            M[c, i, i] = 1 - s
            M[c, i, i - 1] += s
    return M, conv, diag

# >>> Section 9 replaces this line with the real FLEXPART routine <<<
TEACHER = teacher_standin

t0 = time.perf_counter()
M_all, conv_all, diag_all = TEACHER(T_all, q_all, ps_all)
t_teacher = time.perf_counter() - t0
print(f"Teacher: {t_teacher:.1f} s for {len(T_all):,} columns | convective: {conv_all.mean():.1%}")
print("Row sums all 1:", np.allclose(M_all.sum(axis=2), 1.0))

## 4. Features, splits and normalisation
Inputs per column: T profile, q profile (g/kg), surface pressure, CAPE, CIN, LCL pressure
→ `2 × 27 + 4 = 58` features. ERA5 data is split **by time** (train / validation / test
years); synthetic data is split at random.

In [ ]:
def build_features(T, q, ps, diag):
    return np.concatenate([
        T, q * 1000.0, ps[:, None],
        diag["cape"][:, None] / 1000.0, diag["cin"][:, None] / 100.0,
        diag["lcl_p"][:, None] / 1000.0,
    ], axis=1).astype(np.float32)

X_all = build_features(T_all, q_all, ps_all, diag_all)
y_all = conv_all.astype(np.float32)
N = len(X_all)

if times_all is not None:
    years = times_all.astype("datetime64[Y]").astype(int) + 1970
    uy = np.unique(years)
    tr = np.isin(years, uy[:-2]) if len(uy) >= 3 else np.ones(N, bool)
    va = years == uy[-2] if len(uy) >= 3 else np.zeros(N, bool)
    te = years == uy[-1] if len(uy) >= 3 else np.zeros(N, bool)
    if len(uy) < 3:
        print("Fewer than 3 years of ERA5 — falling back to a random split.")
        times_all = None
if times_all is None:
    perm = rng.permutation(N)
    tr, va, te = np.zeros(N, bool), np.zeros(N, bool), np.zeros(N, bool)
    tr[perm[: int(0.8 * N)]] = True
    va[perm[int(0.8 * N): int(0.9 * N)]] = True
    te[perm[int(0.9 * N):]] = True

X_mean = X_all[tr].mean(axis=0)
X_std = X_all[tr].std(axis=0) + 1e-6
Xn = (X_all - X_mean) / X_std
print(f"train {tr.sum():,} | val {va.sum():,} | test {te.sum():,} | features {X_all.shape[1]}")
np.savez(os.path.join(OUT_DIR, "normalisation.npz"), mean=X_mean, std=X_std, p_levels=P_LEVELS)

## 5. The two models

- **Gate:** MLP 58 → 128 → 128 → 1 (logit).
- **Redistribution net:** MLP 58 → 256 → 256 → 256 → 27×27 scores, then
  `log_softmax` along each row → each row is a valid probability distribution.

Loss for model 2: row-wise KL divergence weighted by each level's air mass,
L = Σᵢ wᵢ Σⱼ Mᵢⱼ ln(Mᵢⱼ / M̂ᵢⱼ).

In [ ]:
class GateNet(nn.Module):
    def __init__(self, d_in, h=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(), nn.Linear(h, 1))
    def forward(self, x):
        return self.net(x).squeeze(-1)

class RedistNet(nn.Module):
    def __init__(self, d_in, n=NLEV, h=256):
        super().__init__()
        self.n = n
        self.net = nn.Sequential(nn.Linear(d_in, h), nn.SiLU(), nn.Linear(h, h), nn.SiLU(),
                                 nn.Linear(h, h), nn.SiLU(), nn.Linear(h, n * n))
    def forward(self, x):                       # returns log M̂, shape (B, n, n)
        z = self.net(x).view(-1, self.n, self.n)
        return F.log_softmax(z, dim=-1)

W_LEV = torch.tensor(LAYER_W, dtype=torch.float32, device=DEVICE)

def weighted_kl(logM_hat, M):
    kl_rows = (torch.xlogy(M, M) - M * logM_hat).sum(dim=-1)   # (B, n)
    return (kl_rows * W_LEV).sum(dim=-1).mean()

d_in = X_all.shape[1]
gate, redist = GateNet(d_in).to(DEVICE), RedistNet(d_in).to(DEVICE)
n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"Gate: {n_params(gate):,} params | Redistribution net: {n_params(redist):,} params "
      f"(≈ {4 * (n_params(gate) + n_params(redist)) / 1e6:.2f} MB)")

## 6. Training (checkpoints saved every epoch)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def loader(mask, with_M=False, shuffle=True):
    tensors = [torch.tensor(Xn[mask]), torch.tensor(y_all[mask])]
    if with_M:
        tensors.append(torch.tensor(M_all[mask], dtype=torch.float32))
    return DataLoader(TensorDataset(*tensors), batch_size=CFG["batch_size"], shuffle=shuffle)

def train(model, step_fn, train_dl, val_dl, epochs, name):
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
    ckpt = os.path.join(OUT_DIR, f"{name}.pt")
    best, hist = np.inf, []
    for ep in range(1, epochs + 1):
        model.train(); tl = []
        for batch in train_dl:
            loss = step_fn(model, [b.to(DEVICE) for b in batch])
            opt.zero_grad(); loss.backward(); opt.step(); tl.append(loss.item())
        model.eval()
        with torch.no_grad():
            vl = np.mean([step_fn(model, [b.to(DEVICE) for b in batch]).item() for batch in val_dl])
        sched.step(vl); hist.append((np.mean(tl), vl))
        if vl < best:
            best = vl; torch.save(model.state_dict(), ckpt)
        print(f"[{name}] epoch {ep:3d}  train {np.mean(tl):.5f}  val {vl:.5f}")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return hist

pos_w = torch.tensor((1 - y_all[tr].mean()) / max(y_all[tr].mean(), 1e-6), device=DEVICE)
gate_step = lambda m, b: F.binary_cross_entropy_with_logits(m(b[0]), b[1], pos_weight=pos_w)
redist_step = lambda m, b: weighted_kl(m(b[0]), b[2])

t0 = time.perf_counter()
gate_hist = train(gate, gate_step, loader(tr), loader(va, shuffle=False), CFG["gate_epochs"], "gate")
redist_hist = train(redist, redist_step,
                    loader(tr & conv_all, with_M=True), loader(va & conv_all, with_M=True, shuffle=False),
                    CFG["redist_epochs"], "redist")
print(f"Training time: {(time.perf_counter() - t0) / 60:.1f} min")

## 7. Validation on the test set

In [ ]:
@torch.no_grad()
def predict(Xn_batch):
    x = torch.tensor(Xn_batch, device=DEVICE)
    prob = torch.sigmoid(gate(x)).cpu().numpy()
    M_hat = redist(x).exp().cpu().numpy()
    M_full = np.where(prob[:, None, None] > 0.5, M_hat, np.eye(NLEV)[None])
    return prob, M_hat, M_full

prob_te, Mhat_te, Mfull_te = predict(Xn[te])
y_te, M_te = y_all[te].astype(bool), M_all[te]
pred_te = prob_te > 0.5

# Gate
tp = (pred_te & y_te).sum(); fp = (pred_te & ~y_te).sum(); fn = (~pred_te & y_te).sum()
f1 = 2 * tp / max(2 * tp + fp + fn, 1)

# Matrix error on truly convective columns (gate not involved)
mae_conv = np.abs(Mhat_te[y_te] - M_te[y_te]).mean()
mae_full = np.abs(Mfull_te - M_te).mean()

# Net upward mass flux across each interface k: mass from levels ≤ k ending above k
def upward_flux(M):
    m = LAYER_DP[None, :, None]
    return np.stack([(m[:, : k + 1] * M[:, : k + 1, k + 1:]).sum(axis=(1, 2))
                     for k in range(NLEV - 1)], axis=1)
flux_t, flux_m = upward_flux(M_te[y_te]).mean(0), upward_flux(Mhat_te[y_te]).mean(0)
flux_rel = np.abs(flux_m - flux_t) / np.maximum(flux_t, 1e-9)
flux_max_rel = flux_rel[flux_t > 0.01 * flux_t.max()].max()

# Particle test: final-pressure distribution of mass-weighted particles (1-D Wasserstein)
def final_cdf(M):
    dist = (LAYER_W[None, :, None] * M).sum(axis=1)       # (B, n) mass ending at each level
    return np.cumsum(dist, axis=1)
w1 = (np.abs(final_cdf(Mhat_te[y_te]) - final_cdf(M_te[y_te])) * LAYER_DP[None, :]).sum(axis=1)

# Speed (CPU, same batch). With the stand-in teacher this is NOT the real comparison.
cpu_gate, cpu_redist = GateNet(d_in), RedistNet(d_in)
cpu_gate.load_state_dict(gate.state_dict()); cpu_redist.load_state_dict(redist.state_dict())
xb = torch.tensor(Xn[te][:2000])
t0 = time.perf_counter()
with torch.no_grad():
    _ = cpu_gate(xb); _ = cpu_redist(xb).exp()
t_model = (time.perf_counter() - t0) / len(xb)
t0 = time.perf_counter(); _ = TEACHER(T_all[te][:2000], q_all[te][:2000]); t_teach = (time.perf_counter() - t0) / 2000

results = {
    "gate_F1": float(f1), "gate_TP_FP_FN": [int(tp), int(fp), int(fn)],
    "matrix_MAE_convective": float(mae_conv), "matrix_MAE_full_pipeline": float(mae_full),
    "mass_flux_max_relative_error": float(flux_max_rel),
    "particle_W1_hPa_median": float(np.median(w1)), "particle_W1_hPa_p95": float(np.percentile(w1, 95)),
    "row_sums_equal_1": bool(np.allclose(Mfull_te.sum(axis=2), 1.0, atol=1e-5)),
    "time_per_column_teacher_us": t_teach * 1e6, "time_per_column_model_us": t_model * 1e6,
}
print(json.dumps(results, indent=2))
with open(os.path.join(OUT_DIR, "validation_results.json"), "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
ax[0].plot([h[0] for h in redist_hist], label="train"); ax[0].plot([h[1] for h in redist_hist], label="validation")
ax[0].set(title="Redistribution net loss (weighted KL)", xlabel="epoch", yscale="log"); ax[0].legend()
ax[1].plot(flux_t, P_LEVELS[:-1], "k-", label="teacher"); ax[1].plot(flux_m, P_LEVELS[:-1], "C1--", label="emulator")
ax[1].invert_yaxis(); ax[1].set(title="Mean upward mass flux", xlabel="hPa of air crossing level", ylabel="pressure (hPa)"); ax[1].legend()
ax[2].hist(w1, bins=40, color="C0"); ax[2].set(title="Particle test: Wasserstein distance", xlabel="hPa", ylabel="columns")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "validation_summary.png"), dpi=130); plt.show()

idx = np.where(y_te)[0][np.argsort(-diag_all["cape"][te][y_te])[:1]][0]
fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
for a, mat, ttl in [(ax[0], M_te[idx], "Teacher M"), (ax[1], Mhat_te[idx], "Emulator M̂"),
                    (ax[2], Mhat_te[idx] - M_te[idx], "Difference")]:
    im = a.imshow(mat, origin="lower", cmap="RdBu_r" if ttl == "Difference" else "viridis",
                  vmin=-0.1 if ttl == "Difference" else 0, vmax=0.1 if ttl == "Difference" else mat.max())
    a.set(title=f"{ttl} (deepest test column)", xlabel="to level j", ylabel="from level i"); plt.colorbar(im, ax=a)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "example_matrix.png"), dpi=130); plt.show()

## 8. Save and export to TorchScript (for FLEXPART coupling in Phase 2)
The exported module takes **raw** features (normalisation is built in) and returns the
convection probability and the full redistribution matrix (identity where no convection).

In [ ]:
class ConvectionEmulator(nn.Module):
    def __init__(self, gate, redist, mean, std):
        super().__init__()
        self.gate, self.redist = gate, redist
        self.register_buffer("mean", torch.tensor(mean)); self.register_buffer("std", torch.tensor(std))
        self.register_buffer("eye", torch.eye(redist.n))
    def forward(self, x_raw):
        x = (x_raw - self.mean) / self.std
        prob = torch.sigmoid(self.gate(x))
        M = self.redist(x).exp()
        M = torch.where((prob > 0.5)[:, None, None], M, self.eye.expand_as(M))
        return prob, M

emu = ConvectionEmulator(cpu_gate, cpu_redist, X_mean, X_std).eval()
scripted = torch.jit.trace(emu, torch.tensor(X_all[:4]))
ts_path = os.path.join(OUT_DIR, "convection_emulator_torchscript.pt")
scripted.save(ts_path)
with torch.no_grad():
    p_chk, M_chk = torch.jit.load(ts_path)(torch.tensor(X_all[te][:256]))
print("TorchScript saved:", ts_path)
print("Reloaded model matches:", np.allclose(M_chk.numpy(), Mfull_te[:256], atol=1e-5))

## 9. Replacing the stand-in with FLEXPART's real convection scheme

Only `TEACHER` in Section 3 changes; Sections 4–8 run unchanged.

**Route A — wrap the Fortran routine in Colab (try first)**
1. Download the FLEXPART source from the official FLEXPART website / repository
   (version 10.4 or 11), e.g. `!git clone <repository URL>`.
2. Find the convection code: `!grep -ril "convect" flexpart/src | head`
   (in v10.4 the core routine is `convect43c.f90`; the redistribution matrix is built around it).
3. Write a small Fortran wrapper subroutine that takes one column
   (T, q, pressure on FLEXPART's levels) and returns the n × n mass-fraction matrix,
   copying in only the module variables the routine needs.
4. Compile it into a Python module: `!f2py -c -m flexconv wrapper.f90 convect43c.f90`
   (`gfortran` is available on Colab).
5. Fill in `teacher_flexpart` below and set `TEACHER = teacher_flexpart`.

**Route B — instrument FLEXPART (on a cluster)**
Add a few lines inside FLEXPART's convection call that write each column's inputs and the
resulting matrix to a file during a short simulation, then load those files here as
`T_all`, `q_all`, `M_all`.

Note: FLEXPART runs convection on its own model levels; interpolate between those and the
27 ERA5 pressure levels used here, or retrain on model-level data for Phase 2.

In [ ]:
def teacher_flexpart(T, q, ps=None, p_hpa=P_LEVELS):
    """Template: call the compiled FLEXPART convection wrapper column by column."""
    raise NotImplementedError(
        "Compile the FLEXPART convection wrapper (Section 9, Route A) and call it here, "
        "returning (M, conv, diag) with the same shapes as teacher_standin.")
    # import flexconv
    # diag = parcel_diagnostics(T, q, p_hpa)           # keep: used for input features
    # M = np.stack([flexconv.redistribution(T[c], q[c], p_hpa) for c in range(len(T))])
    # conv = ~np.all(np.isclose(M, np.eye(M.shape[1])[None]), axis=(1, 2))
    # return M, conv, diag

## Next steps
- Replace the stand-in teacher (Section 9) — this is Gate 1 of the plan.
- Switch `CFG["data_source"]` to `"era5"` with at least three years, so the split is by time.
- Add a held-out region (e.g. tropical Atlantic) and compare its error to the training regions.
- Record the validation numbers against the plan's targets.